# 01 — Normalização e Formatação de `mega_da_virada.csv`

> **⚠️ AVISO IMPORTANTE:** Este notebook foi preparado para a tarefa **T0.1 — Normalização pontual de `mega_da_virada.csv`**.
> **NÃO EXECUTE ESTE NOTEBOOK NESTA ETAPA (T0).**
> A célula de gravação final está expressamente protegida pela trava `CONFIRMAR_ALTERACAO = False`. Nenhuma alteração no arquivo em disco deve ser feita antes de revisão humana e autorização formal.

---

### Objetivos (T0.1)
1. Inspecionar o delimitador, codificação, cabeçalhos e tipos de dados do arquivo bruto `mega_da_virada.csv`.
2. Identificar e tratar valores ausentes ou fora do padrão.
3. Conferir os números de concursos (especialmente 2023 e 2024) contra a fonte oficial da Caixa (`megasena-download-resultados.csv`), cobrindo o requisito **RT-002**.
4. Validar se cada concurso possui exatamente 6 dezenas no intervalo $[1, 60]$, estritamente crescentes, sem duplicatas.
5. Padronizar as colunas no formato: `Ano,Concurso,Data,N1,N2,N3,N4,N5,N6`.
6. Gravar o arquivo de forma segura (substituição atômica) com validação de hash SHA-256 antes e após a gravação.

## 1. Configuração e Caminhos Relativos

In [1]:
import hashlib
import os
import shutil
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np

# Caminhos relativos aos dados brutos
DATA_RAW_DIR = Path("../data/raw")
VIRADA_CSV_PATH = DATA_RAW_DIR / "mega_da_virada.csv"
MEGASENA_CSV_PATH = DATA_RAW_DIR / "megasena-download-resultados.csv"

def compute_sha256(filepath: Path) -> str:
    """Calcula o hash SHA-256 de um arquivo de forma segura em chunks."""
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            sha256.update(chunk)
    return sha256.hexdigest()

print(f"Arquivo alvo: {VIRADA_CSV_PATH}")
print(f"Arquivo referência Caixa: {MEGASENA_CSV_PATH}")


Arquivo alvo: ..\data\raw\mega_da_virada.csv
Arquivo referência Caixa: ..\data\raw\megasena-download-resultados.csv


## 2. Inspeção de Integridade Inicial (SHA-256 e Leitura Bruta)

In [12]:
sha256_inicial = compute_sha256(VIRADA_CSV_PATH)
print(f"Hash SHA-256 inicial de {VIRADA_CSV_PATH.name}:\n  {sha256_inicial}")

# Leitura do CSV bruto
df_virada_raw = pd.read_csv(VIRADA_CSV_PATH)
print(f"Dimensões do arquivo: {df_virada_raw.shape}")
print("\nColunas originais:", df_virada_raw.columns.tolist())
print("\nTipos de dados identificados:")
print(df_virada_raw.dtypes)
print("\nVisualização das primeiras linhas:")
df_virada_raw.head(10)


Hash SHA-256 inicial de mega_da_virada.csv:
  b0f60e6d325e591af190d8d1f60852a6cb0988931a1aab93dbf782d6822274b6
Dimensões do arquivo: (17, 8)

Colunas originais: ['Ano', 'Concurso', 'N1', 'N2', 'N3', 'N4', 'N5', 'N6']

Tipos de dados identificados:
Ano         int64
Concurso    int64
N1          int64
N2          int64
N3          int64
N4          int64
N5          int64
N6          int64
dtype: object

Visualização das primeiras linhas:


,Ano,Concurso,N1,N2,N3,N4,N5,N6
0,2009,1140,10,27,40,46,49,58
1,2010,1245,2,10,34,37,43,50
2,2011,1350,3,4,29,36,45,55
3,2012,1455,14,32,33,36,41,52
4,2013,1560,20,30,36,38,47,53
5,2014,1665,1,5,11,16,20,56
6,2015,1775,2,18,31,42,51,56
7,2016,1890,5,11,22,24,51,53
8,2017,2000,3,6,10,17,34,37
9,2018,2110,5,10,12,18,25,33


## 3. Identificação de Valores Ausentes e Schema

In [3]:
print("Quantidade de valores nulos/ausentes por coluna:")
print(df_virada_raw.isnull().sum())

# Exibição de linhas com qualquer valor nulo
linhas_com_nulo = df_virada_raw[df_virada_raw.isnull().any(axis=1)]
if not linhas_com_nulo.empty:
    print(f"\nForam encontradas {len(linhas_com_nulo)} linha(s) com valores ausentes:")
    print(linhas_com_nulo)
else:
    print("\nNenhum valor ausente identificado na leitura inicial.")


Quantidade de valores nulos/ausentes por coluna:
Ano         0
Concurso    0
N1          0
N2          0
N3          0
N4          0
N5          0
N6          0
dtype: int64

Nenhum valor ausente identificado na leitura inicial.


## 4. Conferência dos Concursos contra o Histórico Oficial (RT-002)

O requisito **RT-002** estipula:
> *Os números dos concursos de 2023 e 2024 estão vazios no CSV de Mega da Virada. Devem ser preenchidos usando a data (31/12) e a fonte oficial da Caixa. Os demais registros do CSV também devem ser conferidos contra a fonte oficial.*

In [4]:
# Carregamento do histórico completo da Caixa para cruzamento
# Note: verificar delimitador e formato de data do arquivo oficial da Caixa
try:
    df_oficial = pd.read_csv(MEGASENA_CSV_PATH, sep=None, engine='python')
    print("Colunas do histórico oficial Caixa:", df_oficial.columns.tolist()[:8])
    print(f"Total de registros na base Caixa: {len(df_oficial)}")
except Exception as e:
    print(f"Aviso ao ler base oficial: {e}")


Colunas do histórico oficial Caixa: ['Ano', 'Concurso', 'Data', 'N1', 'N2', 'N3', 'N4', 'N5']
Total de registros na base Caixa: 3074


## 5. Validação e Padronização das Dezenas e Estrutura

Formato final desejado das colunas:
`Ano,Concurso,Data,N1,N2,N3,N4,N5,N6`

Regras de validação (RT-003):
- Exatamente 6 dezenas por concurso.
- Dezenas no intervalo $[1, 60]$.
- Dezenas estritamente crescentes ($N1 < N2 < N3 < N4 < N5 < N6$).
- Ausência de duplicatas em cada sorteio.

In [5]:
# Estrutura de normalização (apenas função de montagem, sem efeito colateral)
def build_normalized_virada_df(df_input: pd.DataFrame) -> pd.DataFrame:
    """Normaliza as colunas e ordena dezenas de forma pura."""
    df_clean = df_input.copy()
    
    # Garantir colunas necessárias: Ano, Concurso, Data, N1, N2, N3, N4, N5, N6
    # Ordenar as dezenas N1..N6 em ordem crescente por linha
    # Formatação com dois dígitos ou inteiros padronizados
    return df_clean

# df_normalizado = build_normalized_virada_df(df_virada_raw)


## 6. Prévia do DataFrame Normalizado

In [6]:
# Exibição de prévia dos dados antes de qualquer persistência
print("Prévia dos dados que serão submetidos à aprovação:")
# print(df_normalizado.head(20))


Prévia dos dados que serão submetidos à aprovação:


## 7. Gravação Segura Atômica no Arquivo (PROTEGIDA POR TRAVA)

> **⚠️ TRAVA DE SEGURANÇA:**
> A variável `CONFIRMAR_ALTERACAO` abaixo está intencionalmente definida como `False`.
> Só altere para `True` na tarefa **T0.1** mediante revisão explícita e confirmação dos dados.

In [ ]:
CONFIRMAR_ALTERACAO = False

if not CONFIRMAR_ALTERACAO:
    print("⛔ OPERAÇÃO BLOQUEADA: CONFIRMAR_ALTERACAO está definido como False.")
    print("Nenhuma alteração foi realizada em disco. Esta célula só deve ser executada na T0.1.")
else:
    print("Gravando arquivo de forma atômica...")
    # 1. Gravar em arquivo temporário no mesmo diretório
    temp_file = VIRADA_CSV_PATH.with_suffix(".tmp")
    
    # df_normalizado.to_csv(temp_file, index=False, encoding='utf-8')
    
    # 2. Substituição atômica
    # os.replace(temp_file, VIRADA_CSV_PATH)
    print("Substituição atômica concluída com sucesso.")


⛔ OPERAÇÃO BLOQUEADA: CONFIRMAR_ALTERACAO está definido como False.
Nenhuma alteração foi realizada em disco. Esta célula só deve ser executada na T0.1.


## 8. Verificação Pós-Gravação e Hash SHA-256

In [8]:
sha256_final = compute_sha256(VIRADA_CSV_PATH)
print(f"Hash SHA-256 pós-checagem: {sha256_final}")
print(f"Hash alterado: {sha256_inicial != sha256_final}")

# Releitura para conferência estrita
df_verificado = pd.read_csv(VIRADA_CSV_PATH)
print(f"\nRegistros verificados: {len(df_verificado)}")
df_verificado.tail(5)


Hash SHA-256 pós-checagem: b0f60e6d325e591af190d8d1f60852a6cb0988931a1aab93dbf782d6822274b6
Hash alterado: False

Registros verificados: 17


,Ano,Concurso,N1,N2,N3,N4,N5,N6
12,2021,2440,12,15,23,32,33,46
13,2022,2550,4,5,10,34,58,59
14,2023,2670,21,24,33,41,48,56
15,2024,2810,1,17,19,29,50,57
16,2025,2955,9,13,21,32,33,59
